# 🧬 AgeWeaver: High-Speed GPU Research Benchmark Runner
**Project:** AgeWeaver — Controllable Bidirectional Facial Age Progression and Regression  
**Hardware:** Google Colab NVIDIA GPU (T4 / A100 / V100) with CUDA & FP16 Acceleration  
**Synchronized Workspace:** `Google Drive > AgeWeaver_Research > 08_Code_Backups > AgeWeaver_Colab`  
**Persistent Outputs:** `03_Generated Images`, `04_Metrics_and_Tables`, `05_Figures_and_Ablations`, `07_Reproducibilty_and_Logs`  

---
### Overview
This notebook runs the complete multi-subject, multi-age controlled experiment matrix directly from Google Drive on GPU in under 5 minutes (approx. 3–5 seconds per generation in FP16), and automatically publishes all outputs into the persistent Google Drive research folders.

## 1. Google Drive Synchronization & Workspace Setup
Mount Google Drive and switch into the synchronized `AgeWeaver_Colab` directory.

In [ ]:
import os
import sys

# 1. Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Running in local Jupyter / non-Colab environment.")

# 2. Configure Google Drive Research Tree Paths
GDRIVE_RESEARCH_ROOT = "/content/drive/MyDrive/AgeWeaver_Research"
PROJECT_DIR = os.path.join(GDRIVE_RESEARCH_ROOT, "08_Code_Backups", "AgeWeaver_Colab")

if IN_COLAB:
    if os.path.exists(PROJECT_DIR):
        os.chdir(PROJECT_DIR)
        print(f"✓ Active workspace set to Google Drive: {os.getcwd()}")
    else:
        raise FileNotFoundError(
            f"Could not locate synchronized project at '{PROJECT_DIR}'. "
            f"Please run '.\\scripts\\sync_to_gdrive.ps1' on your local machine first."
        )
else:
    print(f"Current working directory: {os.getcwd()}")

# 3. Add project root and src to sys.path
if "src" not in sys.path:
    sys.path.insert(0, os.path.abspath("src"))
sys.path.insert(0, os.path.abspath("."))

## 2. GPU Check & Accelerated Dependency Installation
Verify NVIDIA GPU availability and install dependencies from `requirements-colab.txt`.

In [ ]:
# Check CUDA GPU Status
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: Running on CPU. For fast generation, enable GPU via Runtime > Change runtime type > T4 GPU.")

# Install AgeWeaver dependencies
!pip install -q -r requirements-colab.txt
print("✓ Dependencies installed and verified successfully.")

## 3. Pre-Warm Models & Verify Biometric Analyzer
Load Stable Diffusion v1-5 in FP16 on GPU and pre-warm InsightFace buffalo_l suite.

In [ ]:
from ageweaver.config import PipelineConfig
from ageweaver.pipeline import AgeWeaverPipeline
from ageweaver.identity import FaceIdentityAnalyzer

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = "float16" if device == "cuda" else "float32"

print(f"Initializing AgeWeaver diffusion engine on {device} (dtype={dtype})...")
pipe_cfg = PipelineConfig(device=device, torch_dtype=dtype, local_files_only=False)
pipeline = AgeWeaverPipeline(config=pipe_cfg)
pipeline.load_pipeline()

analyzer = FaceIdentityAnalyzer()
analyzer._ensure_loaded()
print(f"✓ Pre-trained diffusion pipeline and ArcFace analyzer loaded successfully on {device}!")

## 4. Verify Face Detection & Reference Identity Provenance
Verify young reference portraits (`ID001_RAJESH_KHANNA`, `ID009_DEEPIKA_PADUKONE`, `data/uploads/HRK.jpeg`).

In [ ]:
from PIL import Image
from ageweaver.dataset import DatasetManager

dm = DatasetManager()
test_identities = ["ID001_RAJESH_KHANNA", "ID009_DEEPIKA_PADUKONE"]

for ident_id in test_identities:
    try:
        img_path = dm.get_image_path(ident_id, auto_download=True)
        img = Image.open(img_path).convert("RGB")
        res = analyzer.analyze_image(img)
        print(f"Subject: {ident_id}")
        print(f"  Path       : {img_path}")
        print(f"  Face Det.  : {res.face_detected} (Confidence: {res.det_score:.3f})")
        print(f"  Est. Age   : {res.estimated_age:.1f} yrs | Gender: {res.gender}")
        print(f"  ArcFace    : 512-d normalized embedding extracted ({res.embedding.shape})\n")
    except Exception as e:
        print(f"Could not verify {ident_id}: {e}")

## 5. Run Full Multi-Age GPU Benchmark
Executes age progression and regression across target ages **[2, 12, 35, 50, 70]** using the adaptive delta controller and DPMSolver++ scheduler with Karras sigmas.

In [ ]:
# Execute batch benchmark script
!python scripts/run_colab_benchmark.py \
    --device auto \
    --fp16 \
    --target-ages 2,12,35,50,70 \
    --identities ID001_RAJESH_KHANNA,ID009_DEEPIKA_PADUKONE \
    --custom-images data/uploads/HRK.jpeg \
    --include-synthetic \
    --conditions adaptive_control \
    --steps 15 \
    --scheduler dpmsolver++ \
    --output-dir outputs/colab_benchmark \
    --save-grids

## 6. Render Comparative Visual Grids & Metrics Table
Displays side-by-side progression/regression grids: Input Source vs Target 2, 12, 35, 50, 70.

In [ ]:
import glob
import pandas as pd
from IPython.display import Image as IPyImage, display

# 1. Display comparative visual grids
grid_files = glob.glob("outputs/colab_benchmark/grids/*.png")
print(f"Found {len(grid_files)} comparative visual grids:")
for gf in sorted(grid_files):
    print(f"\nShowing: {os.path.basename(gf)}")
    display(IPyImage(filename=gf))

# 2. Display Quantitative Summary Metrics
csv_path = "outputs/colab_benchmark/experiment_log.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print("\n--- Quantitative Evaluation Summary ---")
    cols = ["experiment_id", "input_id", "source_age", "target_age", "generated_estimated_age", "target_age_mae", "identity_similarity", "runtime_sec"]
    present_cols = [c for c in cols if c in df.columns]
    display(df[present_cols])

## 7. Persist Results into Google Drive Research Folders
Automatically copies newly generated face portraits, quantitative metric tables, visual grids, and execution logs into the canonical Google Drive persistent result folders (`03_Generated Images`, `04_Metrics_and_Tables`, `05_Figures_and_Ablations`, `07_Reproducibilty_and_Logs`).

In [ ]:
import shutil
import glob
import pandas as pd

print("============================================================================")
print(" PERSISTING GPU EXPERIMENT RESULTS TO GOOGLE DRIVE RESEARCH FOLDERS")
print("============================================================================")

if IN_COLAB and os.path.exists(GDRIVE_RESEARCH_ROOT):
    dest_images_dir  = os.path.join(GDRIVE_RESEARCH_ROOT, "03_Generated Images")
    dest_tables_dir  = os.path.join(GDRIVE_RESEARCH_ROOT, "04_Metrics_and_Tables")
    dest_figures_dir = os.path.join(GDRIVE_RESEARCH_ROOT, "05_Figures_and_Ablations")
    dest_logs_dir    = os.path.join(GDRIVE_RESEARCH_ROOT, "07_Reproducibilty_and_Logs")
    
    for d in [dest_images_dir, dest_tables_dir, dest_figures_dir, dest_logs_dir]:
        os.makedirs(d, exist_ok=True)
    
    # 1. Copy generated face portraits to '03_Generated Images'
    gen_images = [f for f in glob.glob("outputs/colab_benchmark/*.png") if not os.path.basename(f).startswith("GRID_")]
    for img_p in gen_images:
        shutil.copy2(img_p, dest_images_dir)
    print(f"✓ Persisted {len(gen_images)} face portraits to: {dest_images_dir}")
    
    # 2. Copy metrics table to '04_Metrics_and_Tables'
    metrics_csv = "outputs/colab_benchmark/experiment_log.csv"
    if os.path.exists(metrics_csv):
        shutil.copy2(metrics_csv, os.path.join(dest_tables_dir, "colab_gpu_experiment_log.csv"))
        df = pd.read_csv(metrics_csv)
        summary_md = os.path.join(dest_tables_dir, "COLAB_GPU_EVALUATION_SUMMARY.md")
        df.to_markdown(summary_md, index=False)
        print(f"✓ Persisted quantitative metrics and markdown table to: {dest_tables_dir}")
    
    # 3. Copy visual grids to '05_Figures_and_Ablations'
    grid_images = glob.glob("outputs/colab_benchmark/grids/*.png")
    for gf in grid_images:
        shutil.copy2(gf, dest_figures_dir)
    print(f"✓ Persisted {len(grid_images)} comparative visual grids to: {dest_figures_dir}")
    
    # 4. Copy execution logs to '07_Reproducibilty_and_Logs'
    log_jsonl = "outputs/colab_benchmark/experiment_log.jsonl"
    if os.path.exists(log_jsonl):
        shutil.copy2(log_jsonl, os.path.join(dest_logs_dir, "colab_gpu_experiment_log.jsonl"))
        print(f"✓ Persisted JSONL experiment log to: {dest_logs_dir}")
    
    print("\n[SUCCESS] All Colab GPU experiment deliverables safely persisted into Google Drive.")
else:
    print("Google Drive root not detected or running locally. Local outputs preserved in outputs/colab_benchmark.")